# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

This notebook demonstrates how to load, process, and visualize the dataset using Croissant schema entity references by `@id`, following best practices.

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step retrieves the dataset information and prepares it for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date published: {metadata.datePublished}")
print("Keywords:", ', '.join(metadata.keywords))

## 2. Data Overview
Review available record sets and their fields. All entities will be referenced by their `@id`.

**Note:** Record sets represent tables or collections in the data package. Each record set may include multiple fields referencing columns. We'll enumerate them.

In [ ]:
# List all record sets and their @id
record_sets = dataset.metadata.recordSet

if record_sets:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        field_ids = [f['@id'] for f in rs.get('field', [])]
        print(f"  Fields (@id): {field_ids}")
else:
    print("No record sets found directly in metadata. Attempting to use dynamic record loading...")

# Since the 'recordSet' may be empty or not directly populated, attempt to discover record set IDs from dataset.records()
record_set_ids = dataset.record_set_ids()

print("Available record set @id values:")
for rs_id in record_set_ids:
    print(f"- {rs_id}")
    # Print available field @id for each record set
    schema = dataset.schema(record_set=rs_id)
    print(f"  Fields in {rs_id}: {[f['@id'] for f in schema['fields']]}")

## 3. Data Extraction
Load data from the specific record set(s) into DataFrames. Use the discovered record set and field `@id`s for referencing.

You can inspect each DataFrame to check the columns (fields), which correspond to Croissant `@id` values. Below we extract all record sets into separate DataFrames.

In [ ]:
# Extract all record sets based on discovered record_set_ids
record_set_ids = dataset.record_set_ids()
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}: {df.columns.tolist()}")
    print(df.head(2))
    print()
# Choose the primary record set for further exploration
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id:
    print(f"Sample records from primary record set (@id={primary_record_set_id}):")
    print(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by criterion, normalizing numeric fields, and grouping by demographic or survey variables. Use field `@id` for references.

For this demonstration, we'll:
- Select a numeric field (e.g., GAD-7 score @id)
- Filter records over a threshold
- Normalize the numeric field
- Group data by a demographic variable (e.g., gender @id)

In [ ]:
# Identify a numeric field and a group/demographic field by @id
# For example, suppose the record set includes GAD-7 scores and gender
# Replace below with actual @id as discovered in your overview if different
primary_df = dataframes[primary_record_set_id]

# Example @id values (these MUST be replaced with actual ones as seen above!)
gad7_field_id = 'cr:GAD7_Score'  # replace with real @id from your dataset
gender_field_id = 'cr:Gender'   # replace with real @id from your dataset

# Confirm valid field ids
print("Fields available for EDA:", primary_df.columns.tolist())
if gad7_field_id not in primary_df.columns:
    gad7_field_id = primary_df.columns[0]  # fallback for notebook demo; adjust for real data

# Apply threshold filtering
threshold = 10
filtered_df = primary_df[primary_df[gad7_field_id].astype(float) > threshold]
print(f"Filtered records where {gad7_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{gad7_field_id}_normalized"] = (
    filtered_df[gad7_field_id].astype(float) - filtered_df[gad7_field_id].astype(float).mean()
) / filtered_df[gad7_field_id].astype(float).std()
print(f"Normalized {gad7_field_id} values:")
print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

# Group by demographic field
if gender_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(gender_field_id)[gad7_field_id].mean().reset_index()
    print(f"Ave. {gad7_field_id} by {gender_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we use matplotlib to visualize distributions.

In [ ]:
import matplotlib.pyplot as plt

# Visualize distribution of GAD-7 scores
plt.figure(figsize=(8, 5))
primary_df[gad7_field_id].astype(float).hist(bins=15, edgecolor='black')
plt.title(f"Distribution of {gad7_field_id}")
plt.xlabel(gad7_field_id)
plt.ylabel("Frequency")
plt.show()

# Visualize mean GAD-7 by gender (if available)
if gender_field_id in primary_df.columns:
    mean_scores = primary_df.groupby(gender_field_id)[gad7_field_id].mean()
    mean_scores.plot(kind='bar', color='skyblue', edgecolor='black')
    plt.title(f"Mean {gad7_field_id} by {gender_field_id}")
    plt.xlabel(gender_field_id)
    plt.ylabel(f"Mean {gad7_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

- The dataset schema was loaded via Croissant URL and accessed by entity `@id`.
- We extracted record set data, previewed fields, filtered and normalized numeric scores, and grouped by demographic field.
- Visualizations of mental health score distributions and demographic segmentations provided further insight.

For further analysis, explore additional fields, record sets, or join demographic and survey information for broader mental health trends. All entity references should always use their Croissant `@id` to ensure consistency with FAIR principles and dataset provenance.